# c4fairness — worked example (regression, student grades)

Same flow, but the model predicts a **continuous** grade, so the error is the signed
residual `y_true - y_pred` and each cluster is summarised by its **median error** (ANOVA
/ Mann-Whitney significance). Data: `Data/student_performance.csv`. Sensitive kinds:
**binary** `sex_F`, **multi-categorical** `Medu` (mother's education, 0–4), **numeric** `age`.

In [ ]:
import pandas as pd
from c4f.preprocessing import encode_categoricals
from c4f.clustering import cluster
from c4f.cli import _build_sensitive_analysis_list, apply_salient_reconstruction
from c4f.experiments import make_recap
from c4f.result_viz import plot_cluster_recap_heatmap
from IPython.display import Image

df = pd.read_csv("../Data/student_performance.csv")
df[["G1", "G2", "studytime", "absences", "sex_F", "Medu", "age", "y_true", "y_pred"]].head()

## 1. Columns + encode + cluster

`Medu` is an integer code (0–4); we mark it categorical so it is analysed as
categories, not a number. `age` stays continuous.

In [ ]:
regular   = ["G1", "G2", "studytime", "absences"]
sensitive = ["sex_F", "Medu", "age"]
col_lists = {"regular": regular, "sensitive": sensitive, "proxy": [], "special": []}
orig_sensitive = list(sensitive)

dfe, cl, cat_names, mcd, ohe = encode_categoricals(
    df.copy(), col_lists, ["Medu"], "kmeans", distance="euclidean"
)
clustering_cols = cl["regular"] + cl["sensitive"]
res = cluster(dfe[clustering_cols], algorithm="kmeans", distance="euclidean",
              n_clusters=3, random_state=42)
print("clusters:", res.n_clusters, "| silhouette:", round(res.silhouette, 3))

## 2. Continuous error + recap

`error_type="regression"` makes the recap report each cluster's median signed error
(`error_mean`, bias direction) and magnitude (`abs_error_mean`), with ANOVA /
Mann-Whitney significance.

In [ ]:
analysis = _build_sensitive_analysis_list(cl["sensitive"], mcd, orig_sensitive, option="salient")

dfe["residual"] = (df["y_true"] - df["y_pred"]).values
res_df = dfe.copy()
res_df["clusters"] = res.labels
apply_salient_reconstruction(res_df, mcd, orig_sensitive)

recap = make_recap(res_df, clustering_cols, sensitive_cols=analysis,
                   error_col="residual", error_type="regression",
                   feature_matrix=res.feature_matrix,
                   continuous_sensitive_cols=["age"])
recap.round(3)

## 3. Heatmap

In [ ]:
plot_cluster_recap_heatmap(recap.copy(), "student_residual", ".", error_label="residual")
Image("student_residual.png")

## Takeaway

`error_mean` is each cluster's average signed residual (over/under-prediction) and
`abs_error_mean` its magnitude. A cluster with a large `|error| mean` and a significant
`error gap sig.` is where grade predictions are least reliable; its `Medu` / `sex_F` /
`age` columns show which students bear that error.